# Bai Shopping Brain — Kaggle Auto Train → Eval → Promotion
Нужны минимум 500 approved Gold-примеров для обучения и отдельный approved holdout `eval-gold.jsonl` (по умолчанию минимум 50). Добавь их как Kaggle Inputs, включи GPU T4 и нажми Run All. Новый checkpoint считается готовым только если проходит benchmark + promotion gate.


In [ ]:
import torch
assert torch.cuda.device_count() >= 1, 'Нужен GPU'
print(torch.cuda.get_device_name(0))


In [ ]:
!pip -q install -U 'transformers>=4.51,<5' 'peft>=0.15,<1' 'datasets>=3,<5' accelerate bitsandbytes sentencepiece
!rm -rf /kaggle/working/tamdeshevle
!git clone --depth 1 --branch main https://github.com/eneonstudio-dev/tamdeshevle.git /kaggle/working/tamdeshevle
%cd /kaggle/working/tamdeshevle


In [ ]:
from pathlib import Path
import subprocess, sys
root=Path('/kaggle/input')
evals=list(root.rglob('eval-gold.jsonl'))
assert len(evals)==1, f'Нужен ровно один eval-gold.jsonl, найдено {len(evals)}'
golds=[p for p in root.rglob('gold.jsonl') if p!=evals[0]]
assert golds, 'Не найден ни один training gold.jsonl'
cmd=[sys.executable,'teacher-lab/training/kaggle_train_pipeline.py']
for p in sorted(golds): cmd += ['--gold',str(p)]
cmd += ['--eval-gold',str(evals[0]),'--out','/kaggle/working/bai_auto_train']
print('TRAIN GOLD:', *golds, sep='\n- ')
print('EVAL GOLD:',evals[0])
subprocess.check_call(cmd)


In [ ]:
from pathlib import Path
print(Path('/kaggle/working/bai_auto_train/pipeline-manifest.json').read_text())
print('Artifacts:')
for p in Path('/kaggle/working/bai_auto_train').glob('*.zip'): print('-',p)
